# Products of a run

This notebook reads the outputs of a pollux run with `PolluxLoader`: shells, halos, convergence and shear maps, and source catalogs, without the baryonification (dark matter only, `dmo`) and with it (`dmb`). `PolluxLoader` only needs the I/O dependencies of pollux (healsparse, astropy, sacc).

It uses the outputs of the validation run, at nside 2048. Run it first (see the README of the shared space), or point `realization_dir` to another realization.

In [ ]:
import os

# be fair to others on a login node, before importing numba and OpenMP libraries
os.environ['OMP_NUM_THREADS'] = os.environ['NUMBA_NUM_THREADS'] = '8'

import healpy as hp
import matplotlib.pyplot as plt
import numpy as np

from pollux import PolluxLoader

# a realization of the validation run (scripts/run_validation.yaml of the shared space), in your scratch
realization_dir = os.path.expandvars('$PSCRATCH/HACC-HOS-Y1/validation/HACC091_1024/nside2048/seed0')
baryon_tag = 'S19_Mc1e14h_thetaej4.0_proj_cutoff'
catalog_tag = 'poisson_b2.0_nbar1.5_lognormal_bias10pc'

dmo = PolluxLoader(realization_dir, catalog_tag=catalog_tag)
dmb = PolluxLoader(realization_dir, baryon_tag=baryon_tag, catalog_tag=catalog_tag)

## Organisation

A realization (simulation, nside, seed) holds one directory per matter model, `dmo/` and `dmb/{tag}/`, with the same products, plus the halo catalogs and the records of the runs. Files needed only to make the products are in `intermediate/`.

Each product directory has a `config.yaml` with its parameters, inputs and code version. `python -m pollux.io.index ROOT` lists them all.

In [ ]:
import sys

!{sys.executable} -m pollux.io.index {realization_dir} 2>/dev/null

In [ ]:
import yaml

with open(os.path.join(dmb.catalog_dir(), 'config.yaml')) as f:
    config = yaml.safe_load(f)
print(yaml.safe_dump({k: config[k] for k in ('product', 'inputs', 'code_version')}, sort_keys=False))

## Shells

Shells are HealSparse maps of particle counts in ranges of comoving distance, one per snapshot. `load_density` gives the density contrast. Below, the same shell without and with baryonification, and their difference, at a lower resolution.

In [ ]:
ishell = 30
nside_plot = 512

maps = [loader.load_density(ishell).degrade(nside_plot).generate_healpix_map() for loader in (dmo, dmb)]
diff = np.where((maps[0] != hp.UNSEEN) & (maps[1] != hp.UNSEEN), maps[1] - maps[0], hp.UNSEEN)
for i, (m, title) in enumerate(zip(maps + [diff], ['dmo', 'dmb', 'dmb - dmo'])):
    hp.mollview(m, sub=(1, 3, i + 1), title=f'{title}, shell {ishell}', cmap='magma' if i < 2 else 'RdBu_r', min=-1 if i < 2 else -0.1, max=3 if i < 2 else 0.1)

## Halos

Halo catalogs, one per shell, give the positions, redshifts and masses of the halos used by the baryonification.

In [ ]:
haloes = dmo.load_haloes(ishell)
print(f"{len(haloes['ra']):,} halos, columns {list(haloes)}")

mass = haloes['sod_halo_mass']
plt.hist(np.log10(mass[mass > 0]), bins=50, log=True)
plt.xlabel(r'$\log_{10} M_{200c}$ ($M_\odot/h$)');

## Convergence and shear

Convergence maps of the tomographic bins (here the SRD redshift distributions), and the shear maps derived from them.

In [ ]:
ibin = 4
kappa = dmb.load_kappa(ibin).degrade(nside_plot).generate_healpix_map()
gamma1, gamma2 = (g.degrade(nside_plot).generate_healpix_map() for g in dmb.load_shear(ibin))

for i, (m, title) in enumerate(zip([kappa, gamma1, gamma2], [r'$\kappa$', r'$\gamma_1$', r'$\gamma_2$'])):
    hp.mollview(m, sub=(1, 3, i + 1), title=f'{title}, bin {ibin}', cmap='RdBu_r', min=-0.03, max=0.03)

## Source catalogs

One FITS catalog per tomographic bin, with positions, redshifts, the fields interpolated at the galaxies (convergence, shear, density, tidal field), intrinsic alignments and observed shapes. The header records the configuration and the git commit of pollux.

In [ ]:
import fitsio

with dmb.load_source_catalog(0) as hdul:
    header = hdul[1].header
print(f"{header['NAXIS2']:,} galaxies, columns {[header[f'TTYPE{i + 1}'] for i in range(header['TFIELDS'])]}")


def read_sample(loader, ibin, columns, n=200_000, seed=0):
    """Read a random subsample of rows of a catalog (the full catalogs are tens of GB)."""
    with fitsio.FITS(os.path.join(loader.catalog_dir(), f'cat_bin{ibin}.fits')) as f:
        nrows = f[1].get_nrows()
        rows = np.sort(np.random.default_rng(seed).choice(nrows, size=min(n, nrows), replace=False))
        return f[1].read(columns=columns, rows=rows)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
for ibin in range(5):
    data = read_sample(dmb, ibin, ['ra', 'dec', 'z'])
    axes[0].hist(data['z'], bins=100, range=(0, 3), histtype='step', density=True, label=f'bin {ibin}')
    if ibin == 0:
        counts = np.bincount(hp.ang2pix(32, data['ra'], data['dec'], lonlat=True), minlength=hp.nside2npix(32))
axes[0].set(xlabel='z', ylabel='n(z)')
axes[0].legend()
plt.sca(axes[1])
hp.mollview(counts, hold=True, title='galaxies of bin 0 (subsample)', cmap='viridis')